# 📦 Notebook 01: Data Collection & Loading
## Customer Churn Prediction — VS Code / Local Development

### Overview
This notebook handles:
- Loading the Telco Customer Churn dataset from local storage
- Initial data exploration and statistics
- Visualization dashboard using Plotly
- Saving raw data for downstream processing

### Environment Detection
Automatically detects if running on Kaggle or VS Code and adjusts paths accordingly.

In [ ]:
# ════════════════════════════════════════
# CELL 2 — Install Extra Libraries
# ════════════════════════════════════════
!pip install -q plotly kaleido
print("✅ Extra libraries installed!")

In [ ]:
# ════════════════════════════════════════
# CELL 3 — Imports
# ════════════════════════════════════════
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All imports successful!")
print(f"Pandas  : {pd.__version__}")
print(f"Numpy   : {np.__version__}")

In [ ]:
# ════════════════════════════════════════
# CELL 4 — Environment Detection & Path Setup
# ════════════════════════════════════════

def get_data_path():
    """Detect running environment and return appropriate data path."""
    if os.path.exists('/kaggle/input/telco-customer-churn/'):
        print("🏠 Running on Kaggle")
        return '/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv'
    else:
        print("💻 Running on VS Code / Local")
        # Try multiple possible paths for VS Code
        possible_paths = [
            'data/raw/telco_churn.csv',
            '../data/raw/telco_churn.csv',
            '../../data/raw/telco_churn.csv',
            '/workspace/data/raw/telco_churn.csv'
        ]
        for path in possible_paths:
            if os.path.exists(path):
                print(f"   ✅ Found data at: {path}")
                return path
        raise FileNotFoundError("Data file not found in any expected location")

def create_directories():
    """Create necessary directories for the project."""
    dirs_to_create = [
        Path('data/raw'),
        Path('data/processed'),
        Path('notebooks'),
        Path('src'),
        Path('models'),
        Path('app'),
        Path('reports/figures'),
        Path('tests')
    ]
    
    for dir_path in dirs_to_create:
        try:
            dir_path.mkdir(parents=True, exist_ok=True)
            print(f"✅ Created directory: {dir_path}")
        except Exception as e:
            print(f"⚠️ Directory already exists or error creating {dir_path}: {e}")

print("="*60)
print("🔧 Setting up environment...")
print("="*60)

create_directories()
DATA_PATH = get_data_path()
print(f"📁 Data path: {DATA_PATH}")

In [ ]:
# ════════════════════════════════════════
# CELL 5 — Load Dataset
# ════════════════════════════════════════

try:
    df = pd.read_csv(DATA_PATH)
    print("="*60)
    print("✅ Dataset Loaded Successfully!")
    print("="*60)
    print(f"📊 Shape         : {df.shape}")
    print(f"📋 Rows          : {df.shape[0]:,}")
    print(f"📋 Columns       : {df.shape[1]}")
    print(f"💾 Memory Usage  : {df.memory_usage().sum() / 1024:.2f} KB")
    print("="*60)
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    print("💡 Ensure the data file exists at the correct path")
    df = None

In [ ]:
# ════════════════════════════════════════
# CELL 6 — Preview Data
# ════════════════════════════════════════

if df is not None:
    print("📋 First 10 Rows:")
    display(df.head(10))
else:
    print("❌ Cannot preview data - dataset not loaded")

In [ ]:
# ════════════════════════════════════════
# CELL 7 — Dataset Info
# ════════════════════════════════════════

if df is not None:
    print("📋 Dataset Info:")
    print("="*60)
    df.info()
else:
    print("❌ Cannot show info - dataset not loaded")

In [ ]:
# ════════════════════════════════════════
# CELL 8 — Data Types Summary
# ════════════════════════════════════════

if df is not None:
    dtype_df = pd.DataFrame({
        'Column': df.columns,
        'DataType': df.dtypes.values,
        'Non-Null Count': df.notnull().sum().values,
        'Null Count': df.isnull().sum().values,
        'Unique Values': df.nunique().values
    })

    print("\n📊 Complete Column Summary:")
    print("="*80)
    print(dtype_df.to_string(index=False))
else:
    print("❌ Cannot show column summary - dataset not loaded")

In [ ]:
# ════════════════════════════════════════
# CELL 9 — Missing Values Analysis
# ════════════════════════════════════════

if df is not None:
    missing = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df)) * 100

    missing_df = pd.DataFrame({
        'Column': missing.index,
        'Missing Count': missing.values,
        'Missing Percentage': missing_pct.values
    }).sort_values('Missing Percentage', ascending=False)

    print("\n📊 Missing Values Report:")
    print("="*50)
    if missing_df['Missing Count'].sum() == 0:
        print("✅ No missing values found!")
    else:
        display(missing_df[missing_df['Missing Count'] > 0])
else:
    print("❌ Cannot analyze missing values - dataset not loaded")

In [ ]:
# ════════════════════════════════════════
# CELL 10 — Statistical Summary
# ════════════════════════════════════════

if df is not None:
    print("\n📊 Numerical Statistics:")
    display(df.describe().round(2).T)
else:
    print("❌ Cannot show statistics - dataset not loaded")

In [ ]:
# ════════════════════════════════════════
# CELL 11 — Categorical Columns Summary
# ════════════════════════════════════════

if df is not None:
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    print(f"\n📋 Categorical Columns ({len(cat_cols)}):")
    print("="*60)
    for col in cat_cols:
        print(f"\n🔹 {col}:")
        print(f"   Unique Values : {df[col].nunique()}")
        print(f"   Values        : {list(df[col].unique())[:10]}")  # Show first 10
else:
    print("❌ Cannot show categorical summary - dataset not loaded")

In [ ]:
# ════════════════════════════════════════
# CELL 12 — Target Variable Analysis
# ════════════════════════════════════════

if df is not None:
    churn_counts = df['Churn'].value_counts()
    churn_pct = df['Churn'].value_counts(normalize=True) * 100

    print("\n🎯 Target Variable - Churn:")
    print("="*40)
    for val in churn_counts.index:
        bar = "█" * int(churn_pct[val]/2)
        print(f"  {val:<5}: {churn_counts[val]:,} ({churn_pct[val]:.1f}%) {bar}")
else:
    print("❌ Cannot analyze target variable - dataset not loaded")

In [ ]:
# ════════════════════════════════════════
# CELL 13 — Advanced Plotly Dashboard
# ════════════════════════════════════════

if df is not None:
    fig = make_subplots(
        rows=3, cols=3,
        subplot_titles=(
            'Churn Distribution',
            'Churn Percentage',
            'Tenure Distribution',
            'Monthly Charges',
            'Contract Type',
            'Internet Service',
            'Senior Citizen',
            'Payment Method',
            'TotalCharges Distribution'
        ),
        specs=[
            [{"type": "bar"}, {"type": "pie"}, {"type": "histogram"}],
            [{"type": "histogram"}, {"type": "bar"}, {"type": "pie"}],
            [{"type": "bar"}, {"type": "bar"}, {"type": "histogram"}]
        ]
    )

    # Plot 1 — Churn Bar
    fig.add_trace(
        go.Bar(
            x=['No Churn', 'Churn'],
            y=churn_counts.values,
            marker_color=['#2ecc71', '#e74c3c'],
            text=churn_counts.values,
            textposition='outside',
            name='Churn Count'
        ), row=1, col=1
    )

    # Plot 2 — Churn Pie
    fig.add_trace(
        go.Pie(
            labels=['No Churn', 'Churn'],
            values=churn_counts.values,
            marker_colors=['#2ecc71', '#e74c3c'],
            hole=0.4,
            name='Churn %'
        ), row=1, col=2
    )

    # Plot 3 — Tenure Histogram
    fig.add_trace(
        go.Histogram(
            x=df['tenure'],
            nbinsx=30,
            marker_color='#3498db',
            name='Tenure'
        ), row=1, col=3
    )

    # Plot 4 — Monthly Charges
    fig.add_trace(
        go.Histogram(
            x=df['MonthlyCharges'],
            nbinsx=30,
            marker_color='#9b59b6',
            name='Monthly Charges'
        ), row=2, col=1
    )

    # Plot 5 — Contract Type
    contract_counts = df['Contract'].value_counts()
    fig.add_trace(
        go.Bar(
            x=contract_counts.index,
            y=contract_counts.values,
            marker_color=['#e74c3c', '#f39c12', '#2ecc71'],
            text=contract_counts.values,
            textposition='outside',
            name='Contract'
        ), row=2, col=2
    )

    # Plot 6 — Internet Service Pie
    internet_counts = df['InternetService'].value_counts()
    fig.add_trace(
        go.Pie(
            labels=internet_counts.index,
            values=internet_counts.values,
            marker_colors=['#3498db', '#e74c3c', '#95a5a6'],
            hole=0.4,
            name='Internet'
        ), row=2, col=3
    )

    # Plot 7 — Senior Citizen
    senior_counts = df['SeniorCitizen'].value_counts()
    fig.add_trace(
        go.Bar(
            x=['Non-Senior', 'Senior'],
            y=senior_counts.values,
            marker_color=['#3498db', '#e74c3c'],
            text=senior_counts.values,
            textposition='outside',
            name='Senior'
        ), row=3, col=1
    )

    # Plot 8 — Payment Method
    payment_counts = df['PaymentMethod'].value_counts()
    fig.add_trace(
        go.Bar(
            x=payment_counts.values,
            y=payment_counts.index,
            orientation='h',
            marker_color='#1abc9c',
            text=payment_counts.values,
            textposition='outside',
            name='Payment'
        ), row=3, col=2
    )

    # Plot 9 — TotalCharges
    df_temp = df.copy()
    df_temp['TotalCharges'] = pd.to_numeric(
        df_temp['TotalCharges'], errors='coerce'
    )
    fig.add_trace(
        go.Histogram(
            x=df_temp['TotalCharges'].dropna(),
            nbinsx=30,
            marker_color='#e67e22',
            name='Total Charges'
        ), row=3, col=3
    )

    fig.update_layout(
        height=1000,
        title_text="📊 Customer Churn Dataset — Complete Overview Dashboard",
        title_font_size=20,
        showlegend=False,
        template='plotly_dark'
    )

    fig.show()
    print("✅ Dashboard displayed!")
else:
    print("❌ Cannot create dashboard - dataset not loaded")

In [ ]:
# ════════════════════════════════════════
# CELL 14 — Save Raw Data
# ════════════════════════════════════════

if df is not None:
    # Define output path
    if os.path.exists('/kaggle/'):
        output_path = Path('/kaggle/working/data/raw/telco_churn.csv')
    else:
        output_path = Path('data/raw/telco_churn.csv')
    
    # Create directory if needed
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Save to CSV
    df.to_csv(output_path, index=False)
    print(f"✅ Raw data saved successfully to: {output_path}")
    print(f"   File size: {os.path.getsize(output_path) / 1024:.2f} KB")
    
    # Verify
    verify_df = pd.read_csv(output_path)
    print(f"   Verification: {verify_df.shape[0]} rows × {verify_df.shape[1]} columns")
else:
    print("❌ Cannot save data - dataset not loaded")

In [ ]:
# ════════════════════════════════════════
# CELL 15 — Summary Report
# ════════════════════════════════════════

if df is not None:
    print("\n" + "="*60)
    print("📋 NOTEBOOK 01 SUMMARY REPORT")
    print("="*60)
    print(f"  Total Rows          : {df.shape[0]:,}")
    print(f"  Total Columns       : {df.shape[1]}")
    print(f"  Numerical Columns   : {len(df.select_dtypes(include=np.number).columns)}")
    print(f"  Categorical Columns : {len(df.select_dtypes(include='object').columns)}")
    print(f"  Missing Values      : {df.isnull().sum().sum()}")
    print(f"  Duplicate Rows      : {df.duplicated().sum()}")
    print(f"  Churn Rate          : {churn_pct['Yes']:.2f}%")
    print(f"  Non-Churn Rate      : {churn_pct['No']:.2f}%")
    print("="*60)
    print("\n✅ Notebook 01 Complete!")
    print("▶️  Proceed to Notebook 02 — Data Cleaning")
else:
    print("❌ Cannot generate summary - dataset not loaded")